# Knowledge-to-Equation — Playground

A step-by-step, interactive walkthrough of the pipeline. Every cell uses the
**same `src` functions** that `main.py` runs, so what you see here matches the
production run.

Before running: install `requirements.txt`, add your keys to `config/.env`, and
put your dataset + literature PDFs in `data/`.

## 1. Setup & config
Allow nested event loops (LlamaParse/LlamaIndex use asyncio) and load the config
+ API keys.

In [ ]:
import nest_asyncio
nest_asyncio.apply()

from src.config_loader import load_config, require_env

cfg = load_config()
print('Topic:', cfg['topic'])
print('Inputs:', cfg['input_symbols'], '->', cfg['input_columns'])
print('Output:', cfg['output_symbol'], '->', cfg['output_column'])

## 2. Knowledge extraction (RAG)
Parse the literature PDFs, build the index, and run the retrieval query. This is
what the LLM 'reads' before proposing equations.

In [ ]:
from src.knowledge import build_retriever

retrieve = build_retriever(cfg)
knowledge = retrieve()
print(knowledge)

## 3. Dataset & extrapolative split
Low-value region → train/validation; high-value region → held-out test set.

In [ ]:
from src.dataset import load_and_split, split_summary

data = load_and_split(cfg)
print(split_summary(data))
data['train_df'].head()

## 4. Phase 1 — initial equation generation
The LLM proposes equations from the retrieved knowledge; each is fitted and
scored. Lower `phase1_iterations` in the config for a quick test run.

In [ ]:
import openai
from src import modeling

client = openai.OpenAI(api_key=require_env('OPENAI_API_KEY'))

summary_p1, results_p1 = modeling.phase1(client, cfg, retrieve, data)
print('Best Phase 1 equation:\n')
print(results_p1[0]['model_code'])
print('validation R2 =', results_p1[0]['validation_r2'])

### Phase 1 — test on the extrapolative set

In [ ]:
init_mse, init_r2, _ = modeling.evaluate_on_test(results_p1, data)
print('Initial best  -> test MSE:', init_mse, '| test R2:', init_r2)

## 5. Phase 3 — refinement
Feed the best equations back to the LLM and refine until
`(1 - validation_R2) <= r2_threshold` (or the round budget runs out).

In [ ]:
summary_p3, results_p3 = modeling.phase3(
    client, cfg, retrieve, data, summary_p1, list(results_p1)
)
print('Best final equation:\n')
print(results_p3[0]['model_code'])
print('validation R2 =', results_p3[0]['validation_r2'])

In [ ]:
final_mse, final_r2, test_summary = modeling.evaluate_on_test(results_p3, data)
print('Final best -> test MSE:', final_mse, '| test R2:', final_r2)

## 6. (Optional) run the whole thing & write output files
Equivalent to `python main.py`. Writes `initial_equation.txt`,
`final_equation.txt` and `run_summary.txt` into `output/`.

In [ ]:
from src.pipeline import run
result = run()
result['output_dir']